In [15]:
import json
from pathlib import Path

import pandas as pd
import requests

URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
PERU = {
    "minlatitude": -18.5,
    "maxlatitude": 0.0,
    "minlongitude": -81.5,
    "maxlongitude": -68.5,
}
MAGNITUD_MINIMA = 4.0

In [16]:
def consultar(inicio, fin):
    params = {
        "format": "geojson",
        "starttime": inicio,
        "endtime": fin,
        "minmagnitude": MAGNITUD_MINIMA,
        "orderby": "time-asc",
        **PERU,
    }
    respuesta = requests.get(URL, params=params, timeout=120)
    respuesta.raise_for_status()
    return respuesta.json()["features"]


muestra = consultar("2026-08-01", "2026-09-01")
len(muestra)

26

In [17]:
muestra[0]

{'type': 'Feature',
 'properties': {'mag': 4.3,
  'place': '85 km ENE of Alianza Cristiana, Peru',
  'time': 1785690584170,
  'updated': 1786812256040,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/us6000thmt',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000thmt&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'reviewed',
  'tsunami': 0,
  'sig': 284,
  'net': 'us',
  'code': '6000thmt',
  'ids': ',us6000thmt,',
  'sources': ',us,',
  'types': ',origin,phase-data,',
  'nst': 17,
  'dmin': 3.485,
  'rms': 0.56,
  'gap': 100,
  'magType': 'mb',
  'type': 'earthquake',
  'title': 'M 4.3 - 85 km ENE of Alianza Cristiana, Peru'},
 'geometry': {'type': 'Point', 'coordinates': [-75.7595, -3.1069, 140.288]},
 'id': 'us6000thmt'}

In [18]:
def a_dataframe(features):
    filas = []
    for f in features:
        p = f["properties"]
        lon, lat, profundidad = f["geometry"]["coordinates"]
        lugar = p.get("place") or ""
        filas.append({
            "id_usgs": f["id"],
            "tiempo_ms": p["time"],
            "actualizado_ms": p["updated"],
            "magnitud": p["mag"],
            "tipo_magnitud": p["magType"],
            "profundidad_km": profundidad,
            "longitud": lon,
            "latitud": lat,
            "lugar": lugar,
            "pais": lugar.split(",")[-1].strip() if "," in lugar else "Sin dato",
            "significancia": p.get("sig"),
            "tsunami": p.get("tsunami"),
            "estado": p.get("status"),
        })
    return pd.DataFrame(filas)


df = a_dataframe(muestra)
df.head()

,id_usgs,tiempo_ms,actualizado_ms,magnitud,tipo_magnitud,profundidad_km,longitud,latitud,lugar,pais,significancia,tsunami,estado
0,us6000thmt,1785690584170,1786812256040,4.3,mb,140.288,-75.7595,-3.1069,"85 km ENE of Alianza Cristiana, Peru",Peru,284,0,reviewed
1,us6000thpt,1785721344824,1787371964040,5.2,mww,123.990,-75.8360,-8.6087,"71 km ESE of Uchiza, Peru",Peru,418,0,reviewed
2,us6000ti6l,1785901242949,1787364405040,4.2,mb,10.000,-76.9798,-6.6308,"31 km W of San Jose De Sisa, Peru",Peru,271,0,reviewed
3,us6000tip9,1786042641230,1789404710040,5.3,mww,10.000,-75.3956,-12.1655,"4 km NNW of Yanacancha, Peru",Peru,435,0,reviewed
4,us6000tiyv,1786114035372,1787431544040,4.1,mb,139.781,-71.5286,-15.2040,"17 km NNW of Tisco, Peru",Peru,259,0,reviewed


In [19]:
df["fecha_utc"] = pd.to_datetime(df["tiempo_ms"], unit="ms", utc=True)
df["fecha_local"] = df["fecha_utc"].dt.tz_convert("America/Lima")
df[["id_usgs", "tiempo_ms", "fecha_utc", "fecha_local", "profundidad_km"]].head()

,id_usgs,tiempo_ms,fecha_utc,fecha_local,profundidad_km
0,us6000thmt,1785690584170,2026-08-02 17:09:44.170000+00:00,2026-08-02 12:09:44.170000-05:00,140.288
1,us6000thpt,1785721344824,2026-08-03 01:42:24.824000+00:00,2026-08-02 20:42:24.824000-05:00,123.990
2,us6000ti6l,1785901242949,2026-08-05 03:40:42.949000+00:00,2026-08-04 22:40:42.949000-05:00,10.000
3,us6000tip9,1786042641230,2026-08-06 18:57:21.230000+00:00,2026-08-06 13:57:21.230000-05:00,10.000
4,us6000tiyv,1786114035372,2026-08-07 14:47:15.372000+00:00,2026-08-07 09:47:15.372000-05:00,139.781


In [20]:
features = []
for anio in range(1990, 2027):
    lote = consultar(f"{anio}-01-01", f"{anio + 1}-01-01")
    features.extend(lote)
    print(anio, len(lote))

hist = a_dataframe(features)
hist["fecha_utc"] = pd.to_datetime(hist["tiempo_ms"], unit="ms", utc=True)
hist["fecha_local"] = hist["fecha_utc"].dt.tz_convert("America/Lima")
hist["anio"] = hist["fecha_local"].dt.year
len(hist)

1990 142
1991 164
1992 75
1993 85
1994 80
1995 198
1996 243
1997 121
1998 169
1999 110
2000 144
2001 553
2002 183
2003 124
2004 138
2005 296
2006 197
2007 384
2008 196
2009 141
2010 181
2011 174
2012 206
2013 251
2014 262
2015 292
2016 364
2017 283
2018 243
2019 253
2020 250
2021 338
2022 324
2023 276
2024 293
2025 300
2026 174


8207

In [21]:
hist["anio"].value_counts().sort_index()

anio
1990    143
1991    163
1992     75
1993     85
1994     80
1995    198
1996    243
1997    121
1998    169
1999    110
2000    144
2001    553
2002    183
2003    124
2004    138
2005    296
2006    197
2007    384
2008    196
2009    141
2010    181
2011    174
2012    206
2013    251
2014    262
2015    292
2016    365
2017    282
2018    243
2019    253
2020    251
2021    337
2022    324
2023    276
2024    293
2025    300
2026    174
Name: count, dtype: int64

In [22]:
hist[["magnitud", "profundidad_km"]].describe()

,magnitud,profundidad_km
count,8207.000000,8207.000000
mean,4.568064,81.782676
std,0.456846,80.585470
min,4.000000,0.800000
25%,4.300000,33.000000
50%,4.500000,55.200000
75%,4.800000,119.885000
max,8.400000,655.000000


In [23]:
hist["tipo_magnitud"].value_counts(dropna=False)

tipo_magnitud
mb     7028
mwc     382
mww     326
ml      109
mw      101
mwr      96
mwb      79
m        41
md       30
ms       15
Name: count, dtype: int64

In [24]:
hist["pais"].value_counts()

pais
Peru        6692
Ecuador     1094
Sin dato     197
Brazil       100
Bolivia       66
Chile         58
Name: count, dtype: int64

In [25]:
hist.isna().sum()

id_usgs           0
tiempo_ms         0
actualizado_ms    0
magnitud          0
tipo_magnitud     0
profundidad_km    0
longitud          0
latitud           0
lugar             0
pais              0
significancia     0
tsunami           0
estado            0
fecha_utc         0
fecha_local       0
anio              0
dtype: int64

In [26]:
hist["id_usgs"].duplicated().sum()

np.int64(0)

In [27]:
hist["dias_hasta_revision"] = (hist["actualizado_ms"] - hist["tiempo_ms"]) / 86_400_000
print(hist["estado"].value_counts())
print((hist["dias_hasta_revision"] > 1).mean())
hist["dias_hasta_revision"].describe()

estado
reviewed    8207
Name: count, dtype: int64
0.9987815279639333


count     8207.000000
mean      2815.790166
std       2994.879255
min          0.011632
25%         77.176076
50%       2039.925779
75%       4883.704650
max      13117.169590
Name: dias_hasta_revision, dtype: float64

In [28]:
Path("../data/raw").mkdir(parents=True, exist_ok=True)
with open("../data/raw/usgs_exploracion.json", "w", encoding="utf-8") as archivo:
    json.dump(features, archivo)

In [29]:
hist.nlargest(8, "magnitud")[["fecha_local", "magnitud", "tipo_magnitud", "profundidad_km", "lugar"]]

,fecha_local,magnitud,tipo_magnitud,profundidad_km,lugar
1613,2001-06-23 15:33:14.130000-05:00,8.4,mww,33.00,"6 km SSW of Atico, Peru"
3159,2007-08-15 18:40:57.890000-05:00,8.0,mwc,39.00,"41 km SW of San Vicente de Cañete, Peru"
6117,2019-05-26 02:41:15.073000-05:00,8.0,mww,122.57,"78 km NE of Navarro, Peru"
900,1996-11-12 11:59:44.030000-05:00,7.7,mwc,33.00,"60 km SW of Changuillo, Peru"
1858,2001-07-07 04:38:43.520000-05:00,7.6,mwc,33.00,"51 km SW of Punta de Bombón, Peru"
5054,2015-11-24 17:45:38.880000-05:00,7.6,mww,606.21,"155 km WNW of Iñapari, Peru"
5055,2015-11-24 17:50:54.370000-05:00,7.6,mww,620.56,"185 km WNW of Iñapari, Peru"
770,1996-02-21 07:51:01.300000-05:00,7.5,mw,10.00,"123 km WSW of Puerto Santa, Peru"


In [30]:
print((hist["profundidad_km"] == 10).sum())
hist["profundidad_km"].value_counts().head(10)

654


profundidad_km
33.0     1057
10.0      654
35.0      295
100.0      40
150.0      19
36.0       16
20.0       14
45.0       14
28.0       13
48.0       12
Name: count, dtype: int64

In [31]:
reciente = hist[hist["fecha_utc"] >= hist["fecha_utc"].max() - pd.Timedelta(days=365)]
print(len(reciente))
print((reciente["dias_hasta_revision"] > 30).mean())
reciente["dias_hasta_revision"].describe()

251
0.796812749003984


count    251.000000
mean      66.830234
std       31.219731
min        0.011632
25%       68.198950
50%       75.783534
75%       84.720001
max      232.675397
Name: dias_hasta_revision, dtype: float64

In [32]:
hist.loc[hist["pais"] == "Sin dato", "lugar"].value_counts().head(15)

lugar
central Peru                       31
near the coast of central Peru     27
southern Peru                      22
Peru-Ecuador border region         21
Near the coast of southern Peru    19
near the coast of Ecuador          15
northern Peru                      10
Ecuador                             9
Peru-Bolivia border region          8
near the coast of northern Peru     7
Central Peru                        7
Southern Peru                       6
off the coast of northern Peru      5
Northern Peru                       4
Off the coast of southern Peru      2
Name: count, dtype: int64

In [33]:
PAISES = ["Peru", "Ecuador", "Bolivia", "Brazil", "Chile", "Colombia"]


def detectar_pais(lugar):
    texto = lugar.lower()
    for pais in PAISES:
        if pais.lower() in texto:
            return pais
    return "Sin dato"


hist["pais"] = hist["lugar"].map(detectar_pais)
hist["pais"].value_counts()

pais
Peru        6864
Ecuador     1118
Brazil       100
Bolivia       66
Chile         58
Sin dato       1
Name: count, dtype: int64

In [34]:
m5 = hist[hist["magnitud"] >= 5.0]
print(len(m5))
m5["anio"].value_counts().sort_index()

1221


anio
1990     37
1991     38
1992     22
1993     15
1994     26
1995     37
1996     48
1997     26
1998     35
1999     18
2000     21
2001    103
2002     28
2003     25
2004     25
2005     52
2006     27
2007     80
2008     27
2009     30
2010     39
2011     24
2012     21
2013     22
2014     31
2015     28
2016     32
2017     32
2018     23
2019     32
2020     27
2021     33
2022     44
2023     27
2024     34
2025     35
2026     17
Name: count, dtype: int64

In [35]:
PROFUNDIDADES_FIJADAS = {10.0, 33.0, 35.0, 100.0, 150.0}
hist["profundidad_fijada"] = hist["profundidad_km"].isin(PROFUNDIDADES_FIJADAS)
hist["magnitud_momento"] = hist["tipo_magnitud"].str.startswith("mw")
hist[["profundidad_fijada", "magnitud_momento"]].mean()

profundidad_fijada    0.251614
magnitud_momento      0.119898
dtype: float64

In [36]:
hist.groupby("tipo_magnitud")["magnitud"].agg(["count", "min", "max", "mean"]).sort_values("count", ascending=False)

,count,min,max,mean
tipo_magnitud,,,,
mb,7028,4.0,6.1,4.461113
mwc,382,4.8,8.0,5.413089
mww,326,4.6,8.4,5.460123
ml,109,4.0,4.8,4.206422
mw,101,5.0,7.5,5.647525
mwr,96,4.0,5.3,4.466667
mwb,79,5.2,7.5,5.797468
m,41,4.0,4.6,4.160976
md,30,4.0,4.7,4.260000


## Conclusiones de la exploración

Fuente: API FDSN del USGS. 8207 sismos de magnitud 4.0 o mayor dentro del rectángulo
que cubre el Perú, entre enero de 1990 y setiembre de 2026.

**Estructura del formato.** El GeoJSON reparte los datos en dos niveles. La profundidad
no está en `properties` sino como tercer elemento de `coordinates`, después de longitud
y latitud. El campo `time` viene en milisegundos desde 1970 en UTC y hay que convertirlo
declarando la zona, porque de lo contrario todos los eventos se desplazan 5 horas.

**Completitud del catálogo.** El conteo anual de M4+ pasa de unos 80 eventos a comienzos
de los noventa a cerca de 300 en los últimos años. Ese crecimiento no es sísmico sino
instrumental. El conteo anual de M5+ en cambio se mantiene entre 15 y 48 durante los
36 años, sin tendencia. Por eso el análisis de tendencia temporal se hace sobre M5+ en
todo el periodo, y el conjunto M4+ se reserva para la distribución espacial y de
profundidad.

**Picos de 2001 y 2007.** Corresponden a las secuencias de réplicas del sismo de Atico
del 23 de junio de 2001, magnitud 8.4, y del sismo de Cañete del 15 de agosto de 2007,
magnitud 8.0. Son eventos reales y se conservan.

**Profundidades fijadas.** El 25.2% de los registros tiene profundidad en uno de cinco
valores repetidos: 33 km aparece 1057 veces, 10 km aparece 654, 35 km aparece 295, más
100 y 150 km. Son valores por defecto que asigna el USGS cuando la red no logra resolver
la profundidad, no mediciones. Se marcan con `profundidad_fijada` y se excluyen de
cualquier lectura fina del perfil de profundidad.

**Tipos de magnitud.** Aparecen diez escalas distintas. `mb` concentra 7028 registros y
su máximo en todo el periodo es 6.1, mientras que `mww`, con 326 registros, llega a 8.4.
`mb` se satura y deja de crecer en eventos grandes, así que las dos escalas no son
intercambiables. Se conserva `tipo_magnitud` como atributo y se marca con
`magnitud_momento` el 12% de registros medidos en escala de momento.

**Alcance geográfico.** El rectángulo es geometría, no frontera. De los 8207 eventos,
6864 son del Perú y 1342 de Ecuador, Brasil, Bolivia y Chile. Se almacenan todos y el
filtro por país se aplica en el análisis.

**Revisiones de la fuente.** De los 251 eventos del último año, el 79.7% fue revisado
después de 30 días de ocurrido, con mediana de 76 días y máximo de 233. Eso fija la
ventana de solapamiento de la carga incremental en 90 días. Por separado, el campo
`updated` muestra diferencias de hasta 13117 días respecto al evento, lo que indica que
el USGS reprocesa el